# Target Population Weighting and Bayesian Nonparametric Survival Inference

In this notebook, we estimate target-population survival effects using reconstructed individual patient data (IPD) augmented with synthetic baseline covariates.

The goal is to move beyond naive pooled survival comparisons by defining a target population through auxiliary summary information and reweighting the pooled data accordingly.

This notebook:
1. loads the reconstructed IPD with covariates
2. loads auxiliary summaries and benchmark pooled estimates
3. defines a target population
4. computes target-population weights
5. estimates weighted survival effects
6. quantifies uncertainty using a Bayesian nonparametric weighting scheme

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from src.causal.estimands import pooled_survival_difference
from src.causal.pooling import (
    compute_covariate_means,
    reweight_to_target,
    weighted_survival_difference,
)

# Reproducibility
np.random.seed(733)

# Paths
DATA_DIR = Path("../data/processed")

# Load processed inputs
df_cov = pd.read_csv(DATA_DIR / "real_kmgpt_ipd_with_covariates.csv")
combined_aux = pd.read_csv(DATA_DIR / "combined_auxiliary_summaries.csv")
naive_pooled = pd.read_csv(DATA_DIR / "naive_pooled_effect.csv")

print("Loaded df_cov:", df_cov.shape)
print("Loaded combined_aux:", combined_aux.shape)
print("Loaded naive_pooled:", naive_pooled.shape)

df_cov.head()

Loaded df_cov: (776, 18)
Loaded combined_aux: (4, 7)
Loaded naive_pooled: (1, 4)


,trial_id,subgroup,time,event,arm_label,curve,treatment,age_median,male_rate,ecog0_rate,metastatic_rate,biomarker_rate,age,male,ecog0,metastatic,biomarker,stage
0,KEYNOTE-181,high_pdl1,0.731707,1,Chemotherapy,0,0,63.0,0.869,0.401,0.924,0.7,56.369120,1,0,0,0,1.165542
1,KEYNOTE-181,high_pdl1,0.890244,0,Chemotherapy,0,0,63.0,0.869,0.401,0.924,0.7,68.148584,1,1,1,0,1.822921
2,KEYNOTE-181,high_pdl1,1.048780,1,Chemotherapy,0,0,63.0,0.869,0.401,0.924,0.7,58.035384,1,0,1,1,2.330231
3,KEYNOTE-181,high_pdl1,1.243902,1,Chemotherapy,0,0,63.0,0.869,0.401,0.924,0.7,61.334523,1,1,1,1,1.968789
4,KEYNOTE-181,high_pdl1,1.365854,1,Chemotherapy,0,0,63.0,0.869,0.401,0.924,0.7,73.194077,1,0,1,1,2.342307


## Covariates Available for Weighting

The augmented dataset now contains individual-level survival outcomes, treatment assignment, and synthetic baseline covariates informed by trial-level auxiliary summaries.

We next select a small set of covariates to define the target population and construct weighting constraints.

In [2]:
# Inspect available columns and choose weighting covariates

print(df_cov.columns.tolist())

covariates = ["age", "male", "ecog0", "metastatic", "biomarker", "stage"]
t0 = 12.0

print("\nChosen covariates for weighting:", covariates)
print("Evaluation time t0:", t0)

df_cov[covariates].describe()

['trial_id', 'subgroup', 'time', 'event', 'arm_label', 'curve', 'treatment', 'age_median', 'male_rate', 'ecog0_rate', 'metastatic_rate', 'biomarker_rate', 'age', 'male', 'ecog0', 'metastatic', 'biomarker', 'stage']

Chosen covariates for weighting: ['age', 'male', 'ecog0', 'metastatic', 'biomarker', 'stage']
Evaluation time t0: 12.0


,age,male,ecog0,metastatic,biomarker,stage
count,776.000000,776.000000,776.000000,776.000000,776.000000,776.000000
mean,62.987504,0.864691,0.394330,0.796392,0.640464,2.211855
std,8.248887,0.342274,0.489021,0.402941,0.480174,0.450939
min,35.105425,0.000000,0.000000,0.000000,0.000000,0.827052
25%,57.270466,1.000000,0.000000,1.000000,0.000000,1.902646
50%,62.837263,1.000000,0.000000,1.000000,1.000000,2.243770
75%,68.331990,1.000000,1.000000,1.000000,1.000000,2.554033
max,89.357448,1.000000,1.000000,1.000000,1.000000,3.247082


## Target Population Definition

We define a target population using covariate summaries. This represents the population for which we want to estimate the survival effect.

The target is specified through moments of baseline covariates rather than individual-level data, reflecting settings where only aggregate information is available.

This target is slightly healthier than pooled data and has lower biomarker average. It should be different enough to induce reweighting

In [3]:
target_means = pd.Series({
    "age": 60.0,
    "male": 0.85,
    "ecog0": 0.30,
    "metastatic": 0.75,
    "biomarker": 0.60,
    "stage": 2.0,
})

target_means

age           60.00
male           0.85
ecog0          0.30
metastatic     0.75
biomarker      0.60
stage          2.00
dtype: float64